# MPC on the robot: two models

The robot from `RobotSimulationOpenLoopModel_Merged.ipynb` (the plant, fixed) under closed-loop control, with two MPCs that differ only in the model they predict with. Each model is defined in its own notebook and loaded from it by cell id:

* **`MPCModel_GripLimit.ipynb`:** the wheels never slip; friction enters as a grip limit, $|F_i| \le (\mu_s - 3\sigma_\mu) N_i$, affine in the state and input.
* **`MPCModel_Bristle.ipynb`:** the plant's bristle friction with a softer bristle (100 µm), linear once its slowly changing values are held at each step of the horizon. It predicts sliding.

Both see only the measurements $[X, Y, \theta, \omega_{1..4}]$ and the reference, every $T_s = 20$ ms, return four voltages held for $T_s$, and solve a convex QP with OSQP.

In [ ]:
# ===========================================================================
# The robot (plant) and the two MPC models, each loaded from its own notebook
# ===========================================================================
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import osqp
from scipy import sparse
from scipy.linalg import expm

PLANT_NOTEBOOK = "RobotSimulationOpenLoopModel_Merged.ipynb"
plant_cells = {c.get("id"): "".join(c["source"]) for c in json.load(open(PLANT_NOTEBOOK, encoding="utf-8"))["cells"]}
exec(plant_cells["constants"])     # simulation_parameters, GRAVITY
exec(plant_cells["plant"])         # OmniMotorState, OmniRobotSimulation
model_cells = {name: {c.get("id"): "".join(c["source"]) for c in json.load(open(name, encoding="utf-8"))["cells"]}
               for name in ("MPCModel_GripLimit.ipynb", "MPCModel_Bristle.ipynb")}
exec(model_cells["MPCModel_GripLimit.ipynb"]["mpc_model"])   # AffineGripModel: no slip, grip limit
exec(model_cells["MPCModel_Bristle.ipynb"]["mpc_model"])     # BristleModel: bristle friction, 100 um
print(f"Loaded the robot from {PLANT_NOTEBOOK} and the two MPC models")

In [ ]:
"""
MPC parameters
==============

The controller's own settings. The physical values (mass, motor, friction, noise) come from simulation_parameters.
"""

mpc_parameters = {
    # Timing
    'Ts': 0.02,                        # Controller period (s): 50 Hz, each voltage held for Ts/dt plant steps
    'N': 20,                           # Prediction horizon (steps): 0.4 s

    # Grip limit: mu_lim = mu_s - deviation_sigmas * sigma_mu (Coulomb's static friction, less the friction noise)
    'deviation_sigmas': 3.0,           # How many standard deviations of the friction noise to take off mu_s

    # Bristle model
    'bristle_deflection': 100e-6,      # The bristle MPC's bristle deflection (m); the plant uses 2 um

    # Cost weights
    'q_pos': [400.0, 400.0, 40.0],     # Pose error weights [X, Y, theta] (1/m^2, 1/m^2, 1/rad^2)
    'q_vel': [4.0, 4.0, 0.4],          # Velocity error weights, along the robot's axes
    'q_terminal': 5.0,                 # Multiplier on the last step's pose and velocity weights
    'r_du': 0.005,                     # Voltage change weight (1/V^2)

    # Velocity estimate
    'estimator_gain': 1.0,             # Share of the pose prediction error put back into the velocity estimate

    # Solver
    'max_iterations': 800,             # OSQP iteration cap, so the solve always fits Ts (about 10 us each); the best answer so far is used
}

## The grip-limit MPC

Each step: estimate the state, build the no-slip model at the estimate, and solve the QP with the grip limit. Equations in `MPCModel_GripLimit.ipynb`.

In [ ]:
class GripLimitMPC:
    """
    The MPC with an affine grip limit. Sees only the measurements [X, Y, theta, omega_1..4] and the reference.
    """

    def __init__(self, params: dict, mpc_params: dict):
        """
        params: simulation_parameters (the physical values the model is built from)
        mpc_params: mpc_parameters
        """
        self.p = params
        self.mp = mpc_params
        self.Ts = mpc_params['Ts']
        self.N = mpc_params['N']
        self.mu_lim = params['mu_s'] - mpc_params['deviation_sigmas'] * params['sigma_mu']
        self.model = AffineGripModel(params)
        N, n, m = self.N, 6, 4
        self.n_x, self.n_u = n * (N + 1), m * N            # QP variables: x_0..x_N, then u_0..u_{N-1}

        # ------------------------------------------------------------------
        # The QP, set up once in OSQP's form: minimize 1/2 z^T P z + q^T z subject to l <= A z <= u,
        # with z = [x_0, ..., x_N, u_0, ..., u_{N-1}], x = [X, Y, theta, v_x, v_y, theta_dot].
        # Each step changes q, l, u and the values of A.
        # ------------------------------------------------------------------
        x_at = lambda j: j * n
        u_at = lambda j: self.n_x + j * m

        # Cost: tracking of pose and velocity at steps 1..N, and voltage changes
        self.q_track = np.zeros((N, n))
        for j in range(N):
            weight = mpc_params['q_terminal'] if j == N - 1 else 1.0
            self.q_track[j, 0:3] = weight * np.array(mpc_params['q_pos'])
            self.q_track[j, 3:6] = weight * np.array(mpc_params['q_vel'])
        du = np.zeros((N, N))                               # sum over j of (u_j - u_{j-1})^2, with u_{-1} fixed
        for j in range(N):
            du[j, j] += 1.0
            if j > 0:
                du[j - 1, j - 1] += 1.0
                du[j, j - 1] -= 1.0
                du[j - 1, j] -= 1.0
        P = sparse.block_diag((sparse.diags(np.concatenate((np.zeros(n), 2.0 * self.q_track.ravel()))),
                               sparse.kron(2.0 * mpc_params['r_du'] * du, sparse.eye(m))), format='csc')

        # Constraint rows: positions fixed here, values filled in each step
        rows, cols = [], []

        def block(r0, c0, height, width):
            r, c = np.meshgrid(np.arange(height), np.arange(width), indexing='ij')
            rows.append((r0 + r).ravel())
            cols.append((c0 + c).ravel())

        row = 0
        block(row, x_at(0), n, n)                           # x_0 = x_hat
        row += n
        for j in range(N):
            # pose_{j+1} - pose_j - R(theta_bar_j) (S_v v_j + S_u u_j) = R(theta_bar_j) s_0
            # v_{j+1} - A_d v_j - B_d u_j = d
            block(row, x_at(j), n, n)                       # pose_j and v_j
            block(row, x_at(j + 1), n, n)                   # pose_{j+1} and v_{j+1}
            block(row, u_at(j), n, m)
            row += n
        rows.append(row + np.arange(self.n_u))              # |V| <= V_max
        cols.append(self.n_x + np.arange(self.n_u))
        row += self.n_u
        for j in range(N):                                  # grip limit: +-F - mu F_n <= 0, start (v_j) and end (v_{j+1})
            for end in (j, j + 1):
                for sign in (1.0, -1.0):
                    block(row, x_at(end) + 3, m, 3)
                    block(row, u_at(j), m, m)
                    row += m
        rows, cols = np.concatenate(rows), np.concatenate(cols)
        self.n_rows = row
        self.order = np.lexsort((rows, cols))               # column by column, the order OSQP stores the matrix in
        indptr = np.concatenate(([0], np.cumsum(np.bincount(cols, minlength=self.n_x + self.n_u))))
        start_values = self._constraint_values(self.model.affine_model(np.zeros(4), np.zeros(4), 0.0),
                                               (np.eye(3), np.zeros((3, 4)), np.zeros(3), np.eye(3), np.zeros((3, 4)), np.zeros(3)),
                                               np.zeros(N))
        A_start = sparse.csc_matrix((start_values, rows[self.order], indptr), shape=(self.n_rows, self.n_x + self.n_u))
        self.solver = osqp.OSQP()
        self.solver.setup(P, np.zeros(self.n_x + self.n_u), A_start, -np.ones(self.n_rows), np.ones(self.n_rows),
                          verbose=False, warm_starting=True, polishing=False, eps_abs=1e-4, eps_rel=1e-4,
                          max_iter=mpc_params['max_iterations'])

        # Memory between steps
        self.u_applied = np.zeros(m)          # last voltages applied
        self.x_pred = None                    # state the model predicted for now
        self.X_plan = None                    # last plan's states (N+1, 6)
        self.U_plan = None                    # last plan's voltages (N, 4)
        self.failures = 0
        self.capped = 0                       # solves that reached the iteration cap (best answer so far used)
        self.statuses = []
        self.solve_times = []                 # [model, QP, total] per step [s]

    def _constraint_values(self, model, disc, theta_bar):
        """Values of the constraint matrix, in the order its rows and columns were listed, then column by column."""
        A_d, B_d, d, S_v, S_u, s_0 = disc
        N, n, m = self.N, 6, 4
        parts = [np.eye(n).ravel()]
        for j in range(N):
            Rj = np.eye(3)
            c, s = np.cos(theta_bar[j]), np.sin(theta_bar[j])
            Rj[0:2, 0:2] = [[c, -s], [s, c]]
            from_j = np.zeros((n, n))                      # coefficients of [pose_j, v_j]
            from_j[0:3, 0:3] = -np.eye(3)
            from_j[0:3, 3:6] = -Rj @ S_v
            from_j[3:6, 3:6] = -A_d
            from_next = np.eye(n)                          # coefficients of [pose_{j+1}, v_{j+1}]
            from_u = np.vstack((-Rj @ S_u, -B_d))
            parts += [from_j.ravel(), from_next.ravel(), from_u.ravel()]
        parts.append(np.ones(self.n_u))
        for j in range(N):
            for end in (j, j + 1):
                for sign in (1.0, -1.0):
                    parts += [(sign * model['C_F'] - self.mu_lim * model['C_N']).ravel(),
                              (sign * model['D_F'] - self.mu_lim * model['D_N']).ravel()]
        return np.concatenate(parts)[self.order]

    @staticmethod
    def _wrap(angle):
        """Angle into [-pi, pi)."""
        return (angle + np.pi) % (2.0 * np.pi) - np.pi

    @staticmethod
    def _to_robot_axes(theta, vec):
        c, s = np.cos(theta), np.sin(theta)
        return np.array([c * vec[0] + s * vec[1], -s * vec[0] + c * vec[1], vec[2]])

    def estimate(self, measurement: np.ndarray) -> np.ndarray:
        """Pose measured; velocity along the robot's axes from the model's prediction, corrected by the pose error."""
        pose = measurement[:3]
        if self.x_pred is None:
            return np.concatenate((pose, np.zeros(3)))     # starts at rest
        error = pose - self.x_pred[:3]
        error[2] = self._wrap(error[2])
        v = self.x_pred[3:6] + self.mp['estimator_gain'] * self._to_robot_axes(pose[2], error) / self.Ts
        return np.concatenate((pose, v))

    def step(self, t: float, measurement: np.ndarray, reference) -> np.ndarray:
        """
        One controller step.

        t: time now [s]
        measurement: [X, Y, theta, omega_1..4] from the robot's measure
        reference: function of time giving [X, Y, theta, x_dot, y_dot, theta_dot] (global)

        Returns the four voltages to hold until the next step [V].
        """
        start = time.perf_counter()
        N, Ts, n, m = self.N, self.Ts, 6, 4
        x_hat = self.estimate(measurement)

        # The model for this step: held values at the estimate, then exact zero-order hold
        model = self.model.affine_model(*self.model.held_values(x_hat[3:6]))
        disc = self.model.discretize(model, Ts)
        A_d, B_d, d, S_v, S_u, s_0 = disc

        # Planned heading for the pose geometry: the last plan's, moved on one step
        if self.X_plan is None:
            theta_bar = x_hat[2] + x_hat[5] * Ts * np.arange(N)
        else:
            theta_bar = np.concatenate(([x_hat[2]], self.X_plan[2:N + 1, 2] - self.X_plan[1, 2] + x_hat[2]))
        built = time.perf_counter()

        # Reference: pose (heading unwrapped next to the measured one) and velocity along the robot's axes
        ref = np.array([reference(t + Ts * (j + 1)) for j in range(N)])
        ref[:, 2] = np.unwrap(ref[:, 2])
        ref[:, 2] -= 2.0 * np.pi * np.round((ref[0, 2] - x_hat[2]) / (2.0 * np.pi))
        target = ref.copy()
        for j in range(N):
            target[j, 3:6] = self._to_robot_axes(ref[j, 2], ref[j, 3:6])

        # QP data for this step
        q = np.concatenate((np.zeros(n), -2.0 * (self.q_track * target).ravel(), np.zeros(self.n_u)))
        q[self.n_x:self.n_x + m] = -2.0 * self.mp['r_du'] * self.u_applied
        lower, upper = [x_hat], [x_hat]
        for j in range(N):
            c, s = np.cos(theta_bar[j]), np.sin(theta_bar[j])
            pose_shift = np.array([c * s_0[0] - s * s_0[1], s * s_0[0] + c * s_0[1], s_0[2]])
            rhs = np.concatenate((pose_shift, d))
            lower.append(rhs)
            upper.append(rhs)
        lower.append(-self.p['V_max'] * np.ones(self.n_u))
        upper.append(self.p['V_max'] * np.ones(self.n_u))
        for j in range(N):                                  # +-F - mu F_n <= mu e_N -+ e_F
            for end in (j, j + 1):
                for sign in (1.0, -1.0):
                    lower.append(np.full(m, -np.inf))
                    upper.append(self.mu_lim * model['e_N'] - sign * model['e_F'])
        self.solver.update(q=q, l=np.concatenate(lower), u=np.concatenate(upper),
                           Ax=self._constraint_values(model, disc, theta_bar))
        if self.X_plan is not None:
            # Start from the last plan moved on one step
            X_ws = np.vstack((x_hat, self.X_plan[2:], self.X_plan[-1:]))
            U_ws = np.vstack((self.U_plan[1:], self.U_plan[-1:]))
            self.solver.warm_start(x=np.concatenate((X_ws.ravel(), U_ws.ravel())))
        result = self.solver.solve()
        self.statuses.append(result.info.status)

        if result.info.status in ('solved', 'solved inaccurate', 'maximum iterations reached') and np.all(np.isfinite(result.x)):
            self.capped += result.info.status == 'maximum iterations reached'
            self.X_plan = result.x[:self.n_x].reshape(N + 1, n)
            self.U_plan = result.x[self.n_x:].reshape(N, m)
            self.u_applied = np.clip(self.U_plan[0], -self.p['V_max'], self.p['V_max'])
        else:
            self.failures += 1                    # hold the last voltages, start the next plan afresh
            self.X_plan = self.U_plan = None

        # What the model expects at the next step, for the next estimate
        v_next = A_d @ x_hat[3:6] + B_d @ self.u_applied + d
        s = S_v @ x_hat[3:6] + S_u @ self.u_applied + s_0
        c, sn = np.cos(theta_bar[0]), np.sin(theta_bar[0])
        self.x_pred = np.concatenate((x_hat[:3] + np.array([c * s[0] - sn * s[1], sn * s[0] + c * s[1], s[2]]), v_next))
        self.last_model = model
        end = time.perf_counter()
        self.solve_times.append((built - start, end - built, end - start))
        return self.u_applied.copy()

## The bristle MPC

Same QP as the grip-limit MPC, on the bristle model: the values are held at each step of the horizon at the state the last plan predicted for it (step 0 at the estimate), and the bristle deflections are corrected each step from the wheels' torque balance.

**The bound $|z_i| \le 0.9$, and why.** The plant's friction is $F_i = -\mu_s N_i z_i$, so $|z_i| = |F_i| / (\mu_s N_i)$ is the share of a wheel's static grip limit in use, and the friction it uses is $\mu_s |z_i|$ times its normal force.

* **Why 0.9:** it is the same grip as the grip-limit MPC, $\mu_s - 3\sigma_\mu = 0.81$, written as a share of $\mu_s$: $0.81 / 0.9 = 0.9$. Taking the friction noise's 3σ off $\mu_s$ leaves room for the floor being less grippy than nominal at that moment.
* **What it does:** the plan never asks a wheel for more friction than $0.81 N_i$. Without it, the plan asked for friction the wheels did not have and the tracking failed (281 mm RMS on the 3 s eight, against 15 mm with it).
* **What it does not do: it does not stop sliding.** A sliding wheel settles at $|z_i| = \mu_{d,i}/\mu_s$, below 0.9 at most slip speeds (about 0.79 at 0.5 m/s of slip), so the bound allows sliding. That is on purpose here: this MPC is meant to predict sliding and use it, not avoid it. To avoid sliding, the limit would be on the slip $v_{s,i} = (\mathbf{H}_{drive}\mathbf{v})_i - r\omega_i$ instead, which is also affine in the state.

It is a bound on a state, so it is exactly affine and the QP stays convex.

In [ ]:
class BristleMPC:
    """
    MPC on the bristle model (BristleModel). Sees only the measurements [X, Y, theta, omega_1..4] and the reference.
    """

    def __init__(self, params: dict, mpc_params: dict):
        self.p = params
        self.mp = mpc_params
        self.Ts = mpc_params['Ts']
        self.N = mpc_params['N']
        self.model = BristleModel(params, mpc_params['bristle_deflection'])
        # Bound on the bristle state, |z| <= z_max. |z| = |F| / (mu_s F_n) is the share of the static grip limit in use,
        # so z_max = (mu_s - 3 sigma_mu) / mu_s = 0.9 caps the friction the plan may ask for at (mu_s - 3 sigma_mu) F_n = 0.81 F_n,
        # the same grip as the grip-limit MPC. It limits friction used, not sliding: a sliding wheel sits at
        # |z| = mu_d / mu_s < 1, so the plan may still slide (on purpose: this MPC predicts and uses sliding).
        self.z_max = (params['mu_s'] - mpc_params['deviation_sigmas'] * params['sigma_mu']) / params['mu_s']
        N, n, m = self.N, 14, 4
        self.n_x, self.n_u = n * (N + 1), m * N
        x_at = lambda j: j * n
        u_at = lambda j: self.n_x + j * m

        # Cost: pose and velocity (along the robot's axes) at steps 1..N, and voltage changes
        self.q_track = np.zeros((N, n))
        for j in range(N):
            weight = mpc_params['q_terminal'] if j == N - 1 else 1.0
            self.q_track[j, 0:3] = weight * np.array(mpc_params['q_pos'])
            self.q_track[j, 3:6] = weight * np.array(mpc_params['q_vel'])
        du = np.zeros((N, N))
        for j in range(N):
            du[j, j] += 1.0
            if j > 0:
                du[j - 1, j - 1] += 1.0
                du[j, j - 1] -= 1.0
                du[j - 1, j] -= 1.0
        P = sparse.block_diag((sparse.diags(np.concatenate((np.zeros(n), 2.0 * self.q_track.ravel()))),
                               sparse.kron(2.0 * mpc_params['r_du'] * du, sparse.eye(m))), format='csc')

        # Constraint rows: x_0 = x_hat, the model at each step, |u| <= V_max, |z| <= z_max
        rows, cols = [], []

        def block(r0, c0, height, width):
            r, c = np.meshgrid(np.arange(height), np.arange(width), indexing='ij')
            rows.append((r0 + r).ravel())
            cols.append((c0 + c).ravel())

        row = 0
        block(row, x_at(0), n, n)
        row += n
        for j in range(N):
            block(row, x_at(j), n, n)
            block(row, x_at(j + 1), n, n)
            block(row, u_at(j), n, m)
            row += n
        rows.append(row + np.arange(self.n_u))
        cols.append(self.n_x + np.arange(self.n_u))
        row += self.n_u
        for j in range(1, N + 1):                         # |z| <= z_max at each predicted step: a bound on the state
            rows.append(row + np.arange(m))
            cols.append(x_at(j) + 10 + np.arange(m))
            row += m
        rows, cols = np.concatenate(rows), np.concatenate(cols)
        self.n_rows = row
        self.order = np.lexsort((rows, cols))
        indptr = np.concatenate(([0], np.cumsum(np.bincount(cols, minlength=self.n_x + self.n_u))))
        identity = (np.tile(np.eye(11), (N, 1, 1)), np.zeros((N, 11, 4)), None, np.zeros((N, 3, 11)), np.zeros((N, 3, 4)), None)
        A_start = sparse.csc_matrix((self._constraint_values(identity, np.zeros(N)), rows[self.order], indptr),
                                    shape=(self.n_rows, self.n_x + self.n_u))
        self.solver = osqp.OSQP()
        self.solver.setup(P, np.zeros(self.n_x + self.n_u), A_start, -np.ones(self.n_rows), np.ones(self.n_rows),
                          verbose=False, warm_starting=True, polishing=False, eps_abs=1e-4, eps_rel=1e-4,
                          max_iter=mpc_params['max_iterations'])

        # Memory between steps
        self.u_applied = np.zeros(m)
        self.x_pred = None
        self.omega_prev = None
        self.X_plan = self.U_plan = None
        self.failures = self.capped = 0
        self.statuses, self.solve_times = [], []

    def _constraint_values(self, disc, theta_bar):
        A_d, B_d, _, S_w, S_u, _ = disc
        n = 14
        parts = [np.eye(n).ravel()]
        for j in range(self.N):
            c, s = np.cos(theta_bar[j]), np.sin(theta_bar[j])
            Rj = np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])
            from_j = np.zeros((n, n))                    # pose_{j+1} = pose_j + R s_j ; w_{j+1} = A_d w_j + B_d u_j + dd
            from_j[0:3, 0:3] = -np.eye(3)
            from_j[0:3, 3:14] = -Rj @ S_w[j]
            from_j[3:14, 3:14] = -A_d[j]
            parts += [from_j.ravel(), np.eye(n).ravel(), np.vstack((-Rj @ S_u[j], -B_d[j])).ravel()]
        parts.append(np.ones(self.n_u))
        parts.append(np.ones(4 * self.N))
        return np.concatenate(parts)[self.order]

    @staticmethod
    def _to_robot_axes(theta, vec):
        c, s = np.cos(theta), np.sin(theta)
        return np.array([c * vec[0] + s * vec[1], -s * vec[0] + c * vec[1], vec[2]])

    def estimate(self, measurement: np.ndarray) -> np.ndarray:
        """Pose and wheel speeds measured; velocity from the prediction corrected by the pose error; bristles from the
        wheels' torque balance over the last Ts."""
        p = self.p
        pose, omega = measurement[:3], measurement[3:7]
        if self.x_pred is None:
            x = np.concatenate((pose, np.zeros(3), omega, np.zeros(4)))
        else:
            error = pose - self.x_pred[:3]
            error[2] = (error[2] + np.pi) % (2.0 * np.pi) - np.pi
            v = self.x_pred[3:6] + self.mp['estimator_gain'] * self._to_robot_axes(pose[2], error) / self.Ts
            x = np.concatenate((pose, v, omega, self.x_pred[10:14]))
            if self.omega_prev is not None:
                _, N, c_roll, _, _ = self.model.held_values(x)
                I = (self.u_applied - p['Kb'] * 0.5 * (omega + self.omega_prev)) / p['R']
                F = (p['Kt'] * I - c_roll * N * p['r'] - p['Jw'] * (omega - self.omega_prev) / self.Ts) / p['r']
                x[10:14] = np.clip(-F / (p['mu_s'] * np.maximum(N, 1e-3)), -1.2, 1.2)
        self.omega_prev = omega.copy()
        return x

    def step(self, t: float, measurement: np.ndarray, reference) -> np.ndarray:
        """One controller step: returns the four voltages to hold until the next step [V]."""
        start = time.perf_counter()
        N, Ts, n, m = self.N, self.Ts, 14, 4
        x_hat = self.estimate(measurement)

        # Held values at each step: the estimate for step 0, then the last plan moved on one step
        points = np.tile(x_hat, (N, 1)) if self.X_plan is None else np.vstack((x_hat, self.X_plan[2:N + 1]))
        As, Bs, cs = (np.array(a) for a in zip(*[self.model.linear_model(self.model.held_values(x)) for x in points]))
        disc = self.model.discretize(As, Bs, cs, Ts)
        A_d, B_d, dd, S_w, S_u, s_0 = disc
        if self.X_plan is None:
            theta_bar = x_hat[2] + x_hat[5] * Ts * np.arange(N)
        else:
            theta_bar = np.concatenate(([x_hat[2]], self.X_plan[2:N + 1, 2] - self.X_plan[1, 2] + x_hat[2]))
        built = time.perf_counter()

        ref = np.array([reference(t + Ts * (j + 1)) for j in range(N)])
        ref[:, 2] = np.unwrap(ref[:, 2])
        ref[:, 2] -= 2.0 * np.pi * np.round((ref[0, 2] - x_hat[2]) / (2.0 * np.pi))
        target = np.zeros((N, n))
        target[:, 0:3] = ref[:, 0:3]
        for j in range(N):
            target[j, 3:6] = self._to_robot_axes(ref[j, 2], ref[j, 3:6])
        q = np.concatenate((np.zeros(n), -2.0 * (self.q_track * target).ravel(), np.zeros(self.n_u)))
        q[self.n_x:self.n_x + m] = -2.0 * self.mp['r_du'] * self.u_applied
        lower, upper = [x_hat], [x_hat]
        for j in range(N):
            c, s = np.cos(theta_bar[j]), np.sin(theta_bar[j])
            rhs = np.concatenate(([c * s_0[j][0] - s * s_0[j][1], s * s_0[j][0] + c * s_0[j][1], s_0[j][2]], dd[j]))
            lower.append(rhs)
            upper.append(rhs)
        lower.append(-self.p['V_max'] * np.ones(self.n_u))
        upper.append(self.p['V_max'] * np.ones(self.n_u))
        lower.append(np.full(4 * N, -self.z_max))
        upper.append(np.full(4 * N, self.z_max))
        self.solver.update(q=q, l=np.concatenate(lower), u=np.concatenate(upper), Ax=self._constraint_values(disc, theta_bar))
        if self.X_plan is not None:
            X_ws = np.vstack((x_hat, self.X_plan[2:], self.X_plan[-1:]))
            U_ws = np.vstack((self.U_plan[1:], self.U_plan[-1:]))
            self.solver.warm_start(x=np.concatenate((X_ws.ravel(), U_ws.ravel())))
        result = self.solver.solve()
        self.statuses.append(result.info.status)
        if result.info.status in ('solved', 'solved inaccurate', 'maximum iterations reached') and np.all(np.isfinite(result.x)):
            self.capped += result.info.status == 'maximum iterations reached'
            self.X_plan = result.x[:self.n_x].reshape(N + 1, n)
            self.U_plan = result.x[self.n_x:].reshape(N, m)
            self.u_applied = np.clip(self.U_plan[0], -self.p['V_max'], self.p['V_max'])
        else:
            self.failures += 1
            self.X_plan = self.U_plan = None

        # What the model expects at the next step
        w = A_d[0] @ x_hat[3:14] + B_d[0] @ self.u_applied + dd[0]
        s = S_w[0] @ x_hat[3:14] + S_u[0] @ self.u_applied + s_0[0]
        c, sn = np.cos(theta_bar[0]), np.sin(theta_bar[0])
        self.x_pred = np.concatenate((x_hat[:3] + [c * s[0] - sn * s[1], sn * s[0] + c * s[1], s[2]], w))
        end = time.perf_counter()
        self.solve_times.append((built - start, end - built, end - start))
        return self.u_applied.copy()

In [ ]:
"""
Closed-loop simulation
======================

The only loop: every Ts the controller gets the measurement and returns voltages, held for Ts/dt plant steps.
The simulation knows the true state; the controller sees only robot.measure.
"""


def figure_eight(t: float, size: float = 0.5, period: float = 4.0, ramp: float = 0.5) -> np.ndarray:
    """
    Figure-eight reference from rest: [X, Y, theta, x_dot, y_dot, theta_dot] at time t (global).

    size: half-width of the eight [m]; period: time for one eight [s]; ramp: time constant of the start [s].
    The heading swings by +-0.5 rad, so the robot also turns.
    """
    rate = 2.0 * np.pi / period
    phase = rate * (t - ramp * (1.0 - np.exp(-t / ramp)))
    phase_dot = rate * (1.0 - np.exp(-t / ramp))
    return np.array([size * np.sin(phase), 0.5 * size * np.sin(2.0 * phase), 0.5 * np.sin(phase),
                     size * np.cos(phase) * phase_dot, size * np.cos(2.0 * phase) * phase_dot, 0.5 * np.cos(phase) * phase_dot])


def run_closed_loop(controller, reference, T: float, params: dict, random_seed: int = 42) -> dict:
    """
    controller: has Ts and step(t, measurement, reference)
    reference: function of time giving [X, Y, theta, x_dot, y_dot, theta_dot]
    T: run length [s]
    params: simulation_parameters of the plant
    random_seed: the plant's noise seed

    Returns a dictionary of arrays at every plant step: time, reference, pose, v_global, omega, I, V, drive slip [m/s]
    and the friction used |z| (1 = mu_s).
    """
    robot = OmniRobotSimulation(params, random_seed=random_seed)
    state = OmniMotorState()
    dt = params['dt']
    hold = int(round(controller.Ts / dt))
    steps = int(round(T / dt))
    log = {name: np.zeros((steps, size)) for name, size in
           (('reference', 6), ('pose', 3), ('v_global', 3), ('omega', 4), ('I', 4), ('V', 4), ('slip', 4), ('z', 4))}
    log['time'] = np.arange(1, steps + 1) * dt
    V = np.zeros(4)
    for k in range(steps):
        t = k * dt
        if k % hold == 0:
            V = controller.step(t, robot.measure(state), reference)
        v_center_d = robot.H_drive @ state.v_local
        state, T_load, S_status = robot.step(state, V)
        log['reference'][k] = reference(t + dt)
        log['pose'][k] = state.pos_global
        log['v_global'][k] = state.v_global
        log['omega'][k] = state.omega
        log['I'][k] = state.I
        log['V'][k] = state.V_actual
        log['slip'][k] = v_center_d - params['r'] * state.omega
        log['z'][k] = state.z
    return log

### Both MPCs on a figure eight

From rest, with the robot's noise on: a figure eight of 0.5 m every 4 s, and the same eight every 3 s, which asks for more grip than the floor gives at some headings. Each run reports the tracking error, the time any wheel slips over 5 mm/s, the most friction a wheel used (share of $\mu_s$), and the solve time against the 20 ms budget.

In [ ]:
# ===========================================================================
# Both MPCs: figure eights at two speeds
# ===========================================================================
controllers = {"grip limit (no slip)": GripLimitMPC, "bristle (100 um)": BristleMPC}
runs = {}
for period in (4.0, 3.0):
    for name, make in controllers.items():
        controller = make(simulation_parameters, mpc_parameters)
        reference = lambda t, period=period: figure_eight(t, period=period)
        log = run_closed_loop(controller, reference, 2.0 * period + 1.0, simulation_parameters, random_seed=42)
        runs[(period, name)] = (controller, log)
        error = 1e3 * np.hypot(*(log['pose'][:, :2] - log['reference'][:, :2]).T)
        times = 1e3 * np.array(controller.solve_times)
        print(f"Eight every {period:.0f} s, {name}: position error RMS {np.sqrt(np.mean(error**2)):.1f} mm (largest {error.max():.1f}) | "
              f"slip {100 * np.mean(np.any(np.abs(log['slip']) > 5e-3, axis=1)):.1f} % of the time, friction used up to {np.abs(log['z']).max():.2f} | "
              f"solve median {np.median(times[:, 2]):.1f} ms, worst {times[:, 2].max():.1f} ms | cap reached {controller.capped}, failed {controller.failures}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for col, period in enumerate((4.0, 3.0)):
    ax = axes[0, col]
    ax.plot(runs[(period, "grip limit (no slip)")][1]['reference'][:, 0], runs[(period, "grip limit (no slip)")][1]['reference'][:, 1], 'k--', label='reference')
    for name in controllers:
        log = runs[(period, name)][1]
        ax.plot(log['pose'][:, 0], log['pose'][:, 1], label=name)
    ax.set_title(f'Path, eight every {period:.0f} s'); ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]'); ax.axis('equal'); ax.legend(); ax.grid(True)
    ax = axes[1, col]
    for name in controllers:
        log = runs[(period, name)][1]
        ax.plot(log['time'], 1e3 * np.hypot(*(log['pose'][:, :2] - log['reference'][:, :2]).T), label=name)
    ax.set_title(f'Position error, eight every {period:.0f} s'); ax.set_xlabel('time [s]'); ax.set_ylabel('[mm]'); ax.legend(); ax.grid(True)
plt.tight_layout()
plt.show()